In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
data_path = Path("/home/coli/Documents/mensabot")
mensas = ["Empa", "Eawag"]

In [ ]:
dfs = []
for mensa in mensas:
    menus_dir = data_path / mensa / "menus"
    if not menus_dir.is_dir():
        raise FileNotFoundError(f"Directory {menus_dir} does not exist.")
    for file in menus_dir.glob(f"*.csv"):
        df = pd.read_csv(file)
        df["Mensa"] = mensa
        dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
df = df_all.dropna(subset=["co2_footprint"])
# Correct vegetarische alternative for burgers
df.loc[(df["vegetarian"] & df["title"].str.lower().str.contains("burger")), "vegetarian"] = False
df = df.loc[df["price"] > 0]  # Filter out zero or negative prices if any

df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set style for better-looking plots
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Create dietary category column
def categorize_diet(row):
    """Categorize menu as vegan, vegetarian, or meat"""
    if row['vegan']:
        return 'Vegan'
    elif row['vegetarian']:
        return 'Vegetarian'
    else:
        return 'Meat'

df['diet_type'] = df.apply(categorize_diet, axis=1)


# Define colors and markers for each combination
colors = {
    "Empa": "#FF6B6B",
    "Eawag": "#4ECDC4",
}

# Display summary
print("Data Summary:")
print(f"Total menus: {len(df)}")
print(f"\nMenus by Mensa:")
print(df['Mensa'].value_counts())
print(f"\nMenus by Diet Type:")
print(df['diet_type'].value_counts())
print(f"\nDiet Type by Mensa:")
print(pd.crosstab(df['Mensa'], df['diet_type']))

In [ ]:
# Price comparison by diet type and mensa
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

# Prepare data for box and bar plots
diets_sorted = sorted(df['diet_type'].unique())
mensas_sorted = sorted(df['Mensa'].unique())
x_positions = np.arange(len(diets_sorted))
bar_width = 0.8 / len(mensas_sorted)

# Box plot using pure matplotlib
for i, mensa in enumerate(mensas_sorted):
    box_data = [df[(df['diet_type'] == d) & (df['Mensa'] == mensa)]['price'].values for d in diets_sorted]
    positions = x_positions + (i - (len(mensas_sorted) - 1) / 2) * bar_width
    bp = axes[0].boxplot(
        box_data,
        positions=positions,
        widths=bar_width * 0.9,
        patch_artist=True,
        medianprops=dict(color='darkred', linewidth=2),
    )
    for patch in bp['boxes']:
        patch.set_facecolor(colors[mensa])
        patch.set_alpha(0.7)

axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(diets_sorted)
axes[0].set_title('Price Distribution by Diet Type and Mensa', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diet Type', fontsize=12)
axes[0].set_ylabel('Price (CHF)', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[mensa], alpha=0.7, label=mensa) for mensa in mensas_sorted]
axes[0].legend(handles=legend_elements)

# Bar plot with mean prices (aligned positions, values, colors, labels)
price_mean = df.groupby(['Mensa', 'diet_type'])['price'].mean().reset_index()
for i, mensa in enumerate(mensas_sorted):
    mensa_vals = [
        price_mean[(price_mean['Mensa'] == mensa) & (price_mean['diet_type'] == diet)]['price'].iloc[0]
        for diet in diets_sorted
    ]
    positions = x_positions + (i - (len(mensas_sorted) - 1) / 2) * bar_width
    axes[1].bar(positions, mensa_vals, bar_width, color=colors[mensa], alpha=0.8, label=mensa)

axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(diets_sorted)
axes[1].set_title('Average Price by Diet Type and Mensa', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diet Type', fontsize=12)
axes[1].set_ylabel('Average Price (CHF)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend()
axes[1].set_ylim(10, None)



plt.tight_layout()
plt.show()

# Print statistics
print("\nPrice Statistics by Mensa and Diet Type:")
print(df.groupby(['Mensa', 'diet_type'])['price'].agg(['count', 'mean', 'median', 'min', 'max']))

In [ ]:
# CO2 Footprint comparison by diet type and mensa
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Prepare data for box and bar plots
diets_sorted = sorted(df['diet_type'].unique())
mensas_sorted = sorted(df['Mensa'].unique())
x_positions = np.arange(len(diets_sorted))
bar_width = 0.8 / len(mensas_sorted)

# Box plot using pure matplotlib
for i, mensa in enumerate(mensas_sorted):
    box_data = [df[(df['diet_type'] == d) & (df['Mensa'] == mensa)]['co2_footprint'].values for d in diets_sorted]
    positions = x_positions + (i - (len(mensas_sorted) - 1) / 2) * bar_width
    bp = axes[0].boxplot(
        box_data,
        positions=positions,
        widths=bar_width * 0.9,
        patch_artist=True,
        medianprops=dict(color='darkred', linewidth=2),
    )
    for patch in bp['boxes']:
        patch.set_facecolor(colors[mensa])
        patch.set_alpha(0.7)

axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(diets_sorted)
axes[0].set_title('CO₂ Footprint Distribution by Diet Type and Mensa', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diet Type', fontsize=12)
axes[0].set_ylabel('CO₂ Footprint (kg CO₂eq)', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[mensa], alpha=0.7, label=mensa) for mensa in mensas_sorted]
axes[0].legend(handles=legend_elements)

# Bar plot with mean CO2 (aligned positions, values, colors, labels)
co2_mean = df.groupby(['Mensa', 'diet_type'])['co2_footprint'].mean().reset_index()
for i, mensa in enumerate(mensas_sorted):
    mensa_vals = [
        co2_mean[(co2_mean['Mensa'] == mensa) & (co2_mean['diet_type'] == diet)]['co2_footprint'].iloc[0]
        for diet in diets_sorted
    ]
    positions = x_positions + (i - (len(mensas_sorted) - 1) / 2) * bar_width
    axes[1].bar(positions, mensa_vals, bar_width, color=colors[mensa], alpha=0.8, label=mensa)

axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(diets_sorted)
axes[1].set_title('Average CO₂ Footprint by Diet Type and Mensa', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diet Type', fontsize=12)
axes[1].set_ylabel('Average CO₂ Footprint (kg CO₂eq)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print("\nCO₂ Footprint Statistics by Mensa and Diet Type:")
print(df.groupby(['Mensa', 'diet_type'])['co2_footprint'].agg(['count', 'mean', 'median', 'min', 'max']))

In [ ]:
# Scatter plot: Price vs CO2 Footprint by Diet Type and Mensa
fig, ax = plt.subplots(figsize=(12, 7))


markers = {"Vegan": "o", "Vegetarian": "s", "Meat": "^"}


# Plot each combination
for mensa in df["Mensa"].unique():
    for diet in df["diet_type"].unique():
        subset = df[
            (df["Mensa"] == mensa) & (df["diet_type"] == diet)
        ]
        ax.scatter(
            subset["price"],
            subset["co2_footprint"],
            label=f"{mensa} - {diet}",
            color=colors[mensa],
            marker=markers[diet],
            s=100,
            alpha=0.6,
            edgecolors="black",
            linewidth=0.5,
        )

ax.set_xlabel("Price (CHF)", fontsize=12, fontweight="bold")
ax.set_ylabel("CO₂ Footprint (kg CO₂eq)", fontsize=12, fontweight="bold")
ax.set_title(
    "Price vs CO₂ Footprint by Mensa and Diet Type", fontsize=14, fontweight="bold"
)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Weekly trend: price and CO2 footprint evolution by mensa
weekly = df.copy()
weekly['date'] = pd.to_datetime(weekly['date'], errors='coerce')
weekly = weekly.dropna(subset=['date'])

weekly['week_start'] = weekly['date'].dt.to_period('W').dt.start_time
weekly_avg = (
    weekly.groupby(['week_start', 'Mensa'])[['price', 'co2_footprint']]
    .mean()
    .reset_index()
    .sort_values('week_start')
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
mensas_sorted = sorted(weekly_avg['Mensa'].unique())

for mensa in mensas_sorted:
    mensa_data = weekly_avg[weekly_avg['Mensa'] == mensa]
    c = colors.get(mensa, None)
    axes[0].plot(
        mensa_data['week_start'],
        mensa_data['price'],
        marker='o',
        linewidth=2,
        color=c,
        label=mensa,
    )
    axes[1].plot(
        mensa_data['week_start'],
        mensa_data['co2_footprint'],
        marker='o',
        linewidth=2,
        color=c,
        label=mensa,
    )

axes[0].set_title('Weekly Average Price by Mensa', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Week Start', fontsize=12)
axes[0].set_ylabel('Price (CHF)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Weekly Average CO₂ Footprint by Mensa', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Week Start', fontsize=12)
axes[1].set_ylabel('CO₂ Footprint (kg CO₂eq)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('Weekly averages (first rows):')
print(weekly_avg.head(10))

In [ ]:
# Best and worst dishes by CO2 footprint for each mensa and food type
from IPython.display import display

co2_extremes = df.copy()
co2_extremes['date'] = pd.to_datetime(co2_extremes['date'], errors='coerce')

cols = ['Mensa', 'diet_type', 'date', 'title', 'description', 'price', 'co2_footprint']
grouped = co2_extremes.groupby(['Mensa', 'diet_type'], group_keys=False)

best_idx = grouped['co2_footprint'].idxmin()
worst_idx = grouped['co2_footprint'].idxmax()

best_dishes = (
    co2_extremes.loc[best_idx, cols]
    .sort_values(['Mensa', 'diet_type'])
    .reset_index(drop=True)
)

worst_dishes = (
    co2_extremes.loc[worst_idx, cols]
    .sort_values(['Mensa', 'diet_type'])
    .reset_index(drop=True)
)

print('Best dishes by mensa and food type (lowest CO₂):')
display(best_dishes)

print('Worst dishes by mensa and food type (highest CO₂):')
display(worst_dishes)

In [ ]:
# Dishes that appear more than once (based on title)
from IPython.display import display

wrong_titles = [
    "Hot&Cold Buffet",
    "Hot & Cold Buffet",
]
this_df = df_all.dropna(subset=['price'])
this_df = this_df.loc[~this_df['title'].isin(wrong_titles)]
title_counts = this_df['title'].value_counts(dropna=False)
duplicate_titles = title_counts[title_counts > 1].index

# 1) DataFrame with only rows whose title appears more than once
repeated_dishes_df = (
    this_df[this_df['title'].isin(duplicate_titles)]
    .sort_values(['title', 'date'])
    .reset_index(drop=True)
)

# 2) DataFrame with duplicate title counts
duplicate_title_counts_df = (
    title_counts[title_counts > 1]
    .rename_axis('title')
    .reset_index(name='count')
    .sort_values(['count', 'title'], ascending=[False, True])
    .reset_index(drop=True)
)

print('Dishes coming more than once (all rows):')
display(repeated_dishes_df)

print('Counts per duplicated dish title:')
display(duplicate_title_counts_df)

In [ ]:
# Histogram "violins": CO2 emissions (left axis) and price (right axis) by diet type and mensa
# Each violin is a mirrored histogram (bar width ∝ count in bin), no KDE smoothing.
from matplotlib.patches import Patch

plt.style.use("default")

fig, ax_co2 = plt.subplots(figsize=(14, 7))
ax_price = ax_co2.twinx()

# Keep category order stable for readability
diets_sorted = sorted(df['diet_type'].dropna().unique())
mensas_sorted = sorted(df['Mensa'].dropna().unique())


df_violin = df.dropna(subset=['co2_footprint', 'price']).copy()

co2_ylim = (0, 6)
price_ylim = (11, 21)

# (column, axis, hatch, alpha, bin width, y-limits)
metrics = [
    ('co2_footprint', ax_co2, None, 0.6, 0.2, co2_ylim),
    ('price', ax_price, '//', 0.6, 0.5, price_ylim),
]
slot_width = 0.3
group_gap = 0.6
slots_per_group = len(metrics) * len(mensas_sorted)
group_width = slots_per_group * slot_width
half_max = slot_width * 0.45  # half-width of the widest bar


def hist_violin(axis, vals, pos, bin_width, color, hatch, alpha):
    """Draw a mirrored histogram centred on x=pos."""
    # Bins centred on multiples of bin_width (avoids rounded values landing on bin edges)
    centers = np.arange(np.round(vals.min() / bin_width), np.round(vals.max() / bin_width) + 1) * bin_width
    edges = np.append(centers - bin_width / 2, centers[-1] + bin_width / 2)
    counts, _ = np.histogram(vals, bins=edges)
    keep = counts > 0  # don't draw empty bins
    half = counts[keep] / counts.max() * half_max
    axis.barh(
        centers[keep], 2 * half, height=bin_width, left=pos - half, align='center',
        color=color, alpha=alpha, edgecolor='black', linewidth=0.5, hatch=hatch,
    )
    axis.hlines(np.mean(vals), pos - half_max, pos + half_max, color='black', lw=1.5)
    axis.hlines(np.median(vals), pos - half_max, pos + half_max, color='darkred', lw=1.5)


xtick_positions = []
for d_idx, diet in enumerate(diets_sorted):
    base = d_idx * (group_width + group_gap)
    xtick_positions.append(base + (slots_per_group - 1) * slot_width / 2)
    for k, (col, axis, hatch, alpha, bin_width, ylim) in enumerate(metrics):
        for m_idx, mensa in enumerate(mensas_sorted):
            vals = pd.to_numeric(
                df_violin[(df_violin['diet_type'] == diet) & (df_violin['Mensa'] == mensa)][col], errors='coerce'
            ).dropna().to_numpy(dtype=float)
            if len(vals) == 0:
                continue
            pos = base + (k * len(mensas_sorted) + m_idx) * slot_width
            hist_violin(axis, vals, pos, bin_width, colors.get(mensa, '#999999'), hatch, alpha)
            # Mensa-coloured arrow above the frame when values extend beyond the visible range
            if (vals > ylim[1]).any():
                axis.annotate(
                    '', xy=(pos, 1.07), xytext=(pos, 1.005),
                    xycoords=axis.get_xaxis_transform(), textcoords=axis.get_xaxis_transform(),
                    annotation_clip=False,
                    arrowprops=dict(arrowstyle='-|>,head_width=0.4,head_length=0.6',
                                    color=colors.get(mensa, '#999999'), lw=2.5),
                )
    # Separator between diet groups
    if d_idx < len(diets_sorted) - 1:
        ax_co2.axvline(base + group_width - slot_width / 2 + group_gap / 2, color='grey', lw=0.8, ls=':')

ax_co2.set_xticks(xtick_positions)
ax_co2.set_xticklabels(diets_sorted)
ax_co2.set_xlim(-slot_width, xtick_positions[-1] + group_width / 2)
dt_start = df_violin['date'].min()
ax_co2.set_title(f'CO₂ Emissions and Price Distribution by Diet Type and Mensa since {dt_start}', fontsize=14, fontweight='bold', pad=12)
ax_co2.set_ylabel('CO₂ Footprint (kg CO₂eq) — solid', fontsize=12)
ax_price.set_ylabel('Price (CHF) — hatched', fontsize=12)
ax_co2.set_ylim(*co2_ylim)
ax_price.set_ylim(*price_ylim)
ax_co2.grid(True, alpha=0.3, axis='y')
ax_co2.grid(False, axis='x')
ax_price.grid(False)

legend_handles = [Patch(facecolor=colors[m], edgecolor='black', alpha=0.6, label=m) for m in mensas_sorted]
legend_handles += [
    Patch(facecolor='white', edgecolor='black', label='CO₂ (left axis)'),
    Patch(facecolor='white', edgecolor='black', hatch='//', label='Price (right axis)'),
    plt.Line2D([], [], color='black', lw=1.5, label='Mean'),
    plt.Line2D([], [], color='darkred', lw=1.5, label='Median'),
]
ax_co2.legend(handles=legend_handles, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# plot the burger pices of empa over sitme
fig, ax = plt.subplots(figsize=(12, 6))
# Use df_all, not df: df drops rows without co2_footprint, which only exists from Feb 2026 on
burger_data = df_all[
    (df_all['Mensa'] == 'Empa') & (df_all['title'].str.lower().str.contains('burger', na=False))
].copy()
burger_data['date'] = pd.to_datetime(burger_data['date'], errors='coerce')
burger_data['price'] = pd.to_numeric(burger_data['price'], errors='coerce')
burger_data = burger_data.dropna(subset=['date', 'price'])
burger_data = burger_data.loc[burger_data['price'] > 0]
burger_data = burger_data.sort_values('date')


ax.plot(burger_data['date'], burger_data['price'], marker='o', linestyle='-', color=colors['Empa'])
ax.set_title('Empa Burger Prices Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price (CHF)', fontsize=12)
ax.grid(True, alpha=0.3)    